# Tissue extractor tutorial (not this)

this is really running the regression on subjects; starting with getting normalization files from SUITPy normalization with respect to MNI Symmetric template, then running the linear regression (on white matter for now; June 10, 2026, 12:35 pm).

# should rename this - this isn't the whole tutorial.

Make this file the file for processing (norm - call once, then convert cell to `raw`; reslice - for all images) for slope, intercept images.

## to do for tutorial

- make the base loop a helper function that you can call for each function

- run each function in its own cell using the helper function

- Above is for the multiple subject case

- Also just show how to do it on one subject

In [ ]:
"""
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl
import ants

import SUITPy as suit
import SUITPy.atlas as atlas

import nitools as nt
import tissue_extractor as te

from pathlib import Path
import os
"""

In [2]:
# if not in same directory as fcn (e.g. avg_vol.py), import cannot find it since notebook is not in root directory of project.
# so add project root
import sys
sys.path.append('/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/')

from image_analysis import avg_vol as av

In [ ]:
# so it doesn't run by mistake
don't run

"""
dummy function to test loop; to be replaced with the actual function
"""
def dummy_fcn(subj_id, week):
    print(subj_id, week)

In [3]:
# directories
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

In [4]:
tissue = 'wm' # default
tissue_dict = {
    'gm': 'c1',
    'wm': 'c2',
    'csf': 'c3'
}

The tissue dict should be treated like a standard run thing, like imports and directories.

In [ ]:
te.normalize?

### step 1

Get the normalization files for MNI symmetric template. (~280 mins)

We want to flip the lesion to the same hemisphere for all subjects. With a symmetric template, we can simply flip about the x-axis.

In [ ]:
# normalization

# this is the base loop - make helper function. Use this loop to run each function of the tissue_extractor in its own cell.

#_______________________________
# base loop
for i in range(0, p_df.shape[0]):
    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = (p_df['RefT1'].iloc[i]).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'

    # check that paths exist
    if not Path(t1_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue

    if not Path(tissue_path).is_file():
        print(f'{tissue} path does not exist for {subj_id} in week {week}')
        continue
    

    # make a new folder inside subject's week folder for results
    results_path = Path(anat_dir)/subj_id/week/'iso_norm_mniSymm/'
    #results_path.mkdir(parents=True, exist_ok = True) # exist_ok = True

    mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'
    #__________________________________

    # function goes here

    te.normalize(t1_path, mask_path, results_path, space = 'MNI152NLin2009cSymC')

    print(f'{subj_id} {week} normalization done')

"""
    results = te.normalize(t1_path = t1_path,
                           mask_path = mask_path,
                           results_path = results_path
                           )

    print(f'{subj_id} {week} normalization done')
"""
    
# run this on one subject, see if it works, stop the loop, upload to repo, and then run all the subjects. Then can upload to repo again.

#### note: for calling te functions
We can use the same base loop for all of them, and then just call the te function. Also for progress update, print which function you used, and done (print(f'{subj_id} {week} normalization done'))

In [ ]:
subj_id = 'CU_2538'
week = 'W0'

results_path = Path(anat_dir)/subj_id/week/'iso_norm_mniSymm/' # folder specifies space

t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'


In [5]:
p_df

,SN,ID,Centre,Week,week,RefT1,numrun,nslices,Hand,LesionSide,...,surfmvpa,include1,lesiondef,behavior_missing,behavior_blocks,has_mvc,DTImap_missing,CentreNo,machine,subj_id
0,1,2310,CU,W0,0,W0,8,35,b,left,...,1,1,1,0,8,0,0,1,naveed,CU_2310
1,2,2310,CU,W4,4,W0,8,35,b,left,...,1,1,1,0,8,0,0,1,naveed,CU_2310
2,3,2310,CU,W12,12,W0,8,35,b,left,...,1,1,1,0,8,1,0,1,naveed,CU_2310
3,4,2310,CU,W24,24,W0,8,35,b,left,...,1,1,1,0,8,1,0,1,naveed,CU_2310
4,5,2310,CU,W52,52,W0,8,35,b,left,...,1,1,1,3,0,1,0,1,naveed,CU_2310
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,210,1008,UZP,W0,0,W0,8,31,b,none,...,1,1,0,0,8,1,1,3,eberlot,UZP_1008
210,211,1008,UZP,W4,4,W0,8,31,b,none,...,1,1,0,0,8,1,1,3,eberlot,UZP_1008
211,212,1008,UZP,W12,12,W0,8,31,b,none,...,1,1,0,0,8,1,1,3,eberlot,UZP_1008
212,213,1008,UZP,W24,24,W0,8,31,b,none,...,1,1,0,0,8,1,1,3,eberlot,UZP_1008


### step 2

Run regression on white matter segmentation in native space (<20 mins).

In [ ]:
# CALL FOR WM SEGMENTATION IMAGE IN NATIVE SPACE________change image suffix for other types

betas = [] # store matrices for all subjects

for subj in p_df['subj_id'].unique():
    #betas.append(avg_vol(subj, ref_img))

    # make directory to store output for each subj
    results_path = Path(anat_dir)/subj/'regression_native/'
    #results_path = f'{anat_dir}/{subj}/regression_native/'
    results_path.mkdir(parents=True, exist_ok = True) # exist_ok = True

    # progress check: note what image is being used
    if tissue:
        print(tissue)

    # fix: use tissue_dict for ref_img
    refT1 = (p_df.loc[(p_df['subj_id']==subj), 'RefT1'].iloc[0]).strip()
    ref_img = f'{anat_dir}/{subj}/{refT1}/{tissue_dict[tissue]}{subj}_{refT1}_T1.nii' # should specify tissue in this call for reference iamge
    # IF NOT USING SEGMENTATION IMAGE (E.G. T1 ANAT), SHOULD CHANGE ABOVE LINE. ALSO EDIT IN THE DICTIONARY TO HAVE OPTION FOR NO TISSUE.

    betas.append(av.avg_vol(subj, 
            reference_img=ref_img, # reference anatomical
            #week_path, # path to week image
            results_path = results_path, # maybe specify folder for this, too (e.g. native_regression)
            image_suffix = f'{tissue}_native', # if using tissue; otherwise, change for T1 anat
            tissue=f'{tissue}'
            ))
    
"""
    betas.append(av.avg_vol(subj_id = subj, reference_img=ref_img,
                         results_path = f'{anat_dir}/{subj}/'),
                         image_suffix = "wm_native"
                         )
"""
    
    # we don't need this weeks loop, the function does it. Really just need to loop through the subject and get their reference img
    # even better if the function can get the reference image itself.
    
    #week = p_df.loc[(p_df['subj_id']==subj), 'week'].iloc[1]
   



# note
In the present avg_vol function, week paths are for c2; this is terrible, need to fix (i.e. have a tissue_dict, where if tissue = None, then it'll just put nothing in front, so like):

tissue_dict = {
    'gm': c1,
    'wm': c2,
    'csf': c3,
    None: '' # i don't know if this notation would work
}

Otherwise, we can just have an if tissue!=None statement, so if tissue is supplied, it'll use the dict above (minus None coding, which is strange and might not work) to find the correct prefix of the file.

I did the latter.

Should check that this works well on one subject before continuing wiht the rest. ANd need to ask Joern about the voxel coordiantes thing first, but also check the code from nilearn.

### step 3

Reslice slope images into template space using MNI symmetric template for normalization.

Note that here, we have exactly one (1) file per subject (that have regression done on them - i.e. >1 measurement weeks).

Our regression was done entirely in the subject's reference image - so it makes sense to use the isolation mask from the reference week (for the slope image reslice). Same for any other reslicing images.

Or would it make more sense to isolate the cerebellum **in the slope image** (or intercept image)?

In [6]:
# reslice

# this is the base loop - make helper function. Use this loop to run each function of the tissue_extractor in its own cell.
#for i in range(0, p_df.shape[0]):
    # p_id = p_df['ID'].iloc[i]
    # #week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces; this is like W<number>
    # p_centre = (str(p_df['Centre'].iloc[i])).strip()
    # refT1 = (p_df['RefT1'].iloc[i]).strip()

    # subj_id = f'{p_centre.strip()}_{p_id}'

    #t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    #tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'

for subj_id in p_df['subj_id'].unique():
    # just need reference week files and file from parent subj directory, so don't need to access all rows.

    refT1 = (p_df.loc[(p_df['subj_id']==subj_id), 'RefT1'].iloc[0]).strip()

    # using (native) wm slope images
    wm_slope_path = f'{anat_dir}/{subj_id}/regression_native/{subj_id}_T1_slope_wm_native.nii.gz'

    # check that paths exist
    # if not Path(t1_path).is_file():
    #     print(f'T1 path does not exist for {subj_id} in week {week}')
    #     continue

    if not Path(wm_slope_path).is_file():
        print(f'{tissue} slope does not exist for {subj_id}')
        continue
    

    # make a new folder inside subject's week folder for results
    results_path = Path(anat_dir)/subj_id/'regression_native/'
    #results_path.mkdir(parents=True) # exist_ok = True

    # for reslicing wm seg slope, use reference T1's isolation mask.
    mask_path = f'{anat_dir}/{subj_id}/{refT1}/iso_norm/{subj_id}_{refT1}_T1_cerebellum_dseg.nii.gz'
    fwd_def = f'{anat_dir}/{subj_id}/{refT1}/iso_norm_mniSymm/{subj_id}_{refT1}_T1_to-MNI152NLin2009cSymC_mode-image_xfm.nii.gz' # for reslice


    te.reslice(
        tissue_path = wm_slope_path,

        # using forward deformation and cerebel isolation mask from reference week
        fwd_def = fwd_def,
        mask_path = mask_path,

        results_path = results_path,
        
        subj_id = subj_id,
        tissue = 'slope_wm_native' # using suffix for slope file; this is just used for naming; NEED TO FIX THIS IN FUNCTION.
    )

    print(f'{subj_id} reslice done')

CU_2310 reslice done
CU_2538 reslice done
CU_2663 reslice done
wm slope does not exist for CU_2697
CU_2925 reslice done
JHU_2282 reslice done
wm slope does not exist for JHU_2374
JHU_2395 reslice done
JHU_2531 reslice done
JHU_2577 reslice done
JHU_2650 reslice done
JHU_2684 reslice done
JHU_2713 reslice done
JHU_2789 reslice done
JHU_3175 reslice done
JHU_3176 reslice done
UZ_2365 reslice done
UZ_2450 reslice done
UZ_2565 reslice done
UZ_2595 reslice done
UZ_2652 reslice done
UZ_2654 reslice done
UZ_2906 reslice done
UZ_3030 reslice done
UZ_3057 reslice done
UZ_3151 reslice done
UZ_3158 reslice done
UZ_3166 reslice done
UZ_3224 reslice done
UZ_3226 reslice done
UZ_3227 reslice done
wm slope does not exist for UZ_3228
UZ_3238 reslice done
UZ_3239 reslice done
UZ_3240 reslice done
UZ_3241 reslice done
UZ_3243 reslice done
UZ_3246 reslice done
UZ_3247 reslice done
UZ_3248 reslice done
CUP_1001 reslice done
CUP_1002 reslice done
JHP_1001 reslice done
JHP_1002 reslice done
JHP_1004 reslice

# to-do for these files

Should rename files with the space that they were resliced to - so use `move_files` function concept but rename files instead of move. (rename as ..._MNISymm_resliced.nii.gz), where we've added <MNISymm>

Also update to `te.reslice`: add option for "space" = <template_name> and use this for suffix on the file name.

Also rename them with the suffix MNISymm_resliced (NOT native in front...that's odd since they're no longer in native).

In [5]:
# reslice all slope images
te.reslice?

Signature:
te.reslice(
    tissue_path,
    fwd_def,
    mask_path,
    results_path,
    subj_id,
    tissue,
    week=None,
)
Docstring: <no docstring>
File:      ~/Documents/GitHub/smarts_cerebellum/image_processing/tissue_extractor.py
Type:      function

### step 4

Choose a lesion side (I'm thinking right side); for subjects with lesions in other hemisphere, flip the lesion to the right hemisphere (along x-axis - hence symmetric template).

For this, we can probably use the base loop but with the participants file cut to only include those participants that need reslice. Note that this base loop will check if the file exists, so those with just one measurement week (who were't run in the regression) won't cause issues.

We can read the participants tsv file as p_df, and call the one with participants whose side needs to be flipped lesion_flip_df or smth

Will flip all lesions to right hemisphere, since more subjects have right hemisphere stroke

**Testing mirror lesion flip function**

In [21]:
from image_processing import mirror_lesion

In [ ]:
# test_subj = 'CU_2310'
# test_image = f'{anat_dir}/{test_subj}/regression_native/{test_subj}_T1_slope_wm_native_resliced.nii.gz'
# flipped_test_img = mirror_lesion.axis_lesion_flip(test_image)
# nib.save(flipped_test_img, f'{anat_dir}/{test_subj}/flipped_test_img.nii.gz')

view in fsleyes: compare this flipped image to the un-flipped image.

In [ ]:
print(len(p_df[p_df.LesionSide=='none '].subj_id.unique()))
print(len(p_df[p_df.LesionSide=='left '].subj_id.unique()))
print(len(p_df[p_df.LesionSide=='right'].subj_id.unique()))

**Most lesions are right hemisphere, so will flip left hemisphere lesion to right hemisphere**

We are flipping the resliced image (normalized to MNI Symmetric template) of the isolated cerebellum for white matter slope.

In [57]:
# flipping left-hem lesion to right hemisphere
left_df = p_df[p_df.LesionSide=='left ']

In [ ]:
# lesion flip

for subj_id in left_df['subj_id'].unique():
    # just need reference week files and file from parent subj directory, so don't need to access all rows.

    # wm slope image resliced to MNI Symm template
    wm_resliced_slope_path = f'{anat_dir}/{subj_id}/regression_native/{subj_id}_T1_slope_wm_native_resliced.nii.gz'

    # check that paths exist
    # if not Path(t1_path).is_file():
    #     print(f'T1 path does not exist for {subj_id} in week {week}')
    #     continue

    if not Path(wm_resliced_slope_path).is_file():
        print(f'{tissue} slope does not exist for {subj_id}')
        continue
    

    # make a new folder inside subject's week folder for results
    results_path = Path(anat_dir)/subj_id/'regression_native/'
    #results_path.mkdir(parents=True) # exist_ok = True

    # function goes here____

    wm_resliced_slope_flipped = mirror_lesion.axis_lesion_flip(wm_resliced_slope_path)

    # save image - probably should put this in the function
    nib.save(wm_resliced_slope_flipped, f'{anat_dir}/{subj_id}/regression_native/{subj_id}_T1_slope_wm_resliced_flipped.nii.gz')
    #_____
    print(f'{subj_id} flipped')

CU_2310 flipped
CU_2663 flipped
wm slope does not exist for CU_2697
CU_2925 flipped
JHU_2282 flipped
wm slope does not exist for JHU_2374
JHU_3175 flipped
UZ_2652 flipped
UZ_2906 flipped
UZ_3166 flipped
UZ_3239 flipped
UZ_3240 flipped
UZ_3243 flipped
UZ_3246 flipped
UZ_3248 flipped


### step 5

Get overall image for patients and controls separately: for all images (with lesion on same side), get_fdata, get in world coordinates (? is necessary?), get average of each voxel in a new matrix, (write matrix back to voxel coordinates), save as nifti. This would be the average slope image for each of patients and controls.

We should be able to get the average of each voxel just like that, since they (each subj's slope image) are all registered to the same template.

Write a function that does this.

So have a separate df for patients and controls. Run the function on each of them separately.